In [2]:
# Copyright (c) CIIS-Lab. All rights reserved.
import os.path as osp
import os
from pathlib import Path
import gc
import copy as cp
import tempfile

import cv2
import mmcv
import mmengine
import numpy as np
import torch

from mmaction.apis import (detection_inference,
                           # inference_recognizer, init_recognizer,
                           pose_inference)
from mmaction.registry import VISUALIZERS
from mmaction.utils import frame_extract

import moviepy.editor as mpy
import glob
import os.path as osp


In [3]:
FONTFACE = cv2.FONT_HERSHEY_DUPLEX
FONTSCALE = 1

THICKNESS = 1  # int
LINETYPE = 1

In [4]:
# def load_label_map(file_path):
#     """Load Label Map.

#     Args:
#         file_path (str): The file path of label map.

#     Returns:
#         dict: The label map (int -> label name).
#     """
#     lines = open(file_path).readlines()
#     lines = [x.strip().split(': ') for x in lines]
#     return {int(x[0]): x[1] for x in lines}


def abbrev(name):
    """Get the abbreviation of label name:

    'take (an object) from (a person)' -> 'take ... from ...'
    """
    while name.find('(') != -1:
        st, ed = name.find('('), name.find(')')
        name = name[:st] + '...' + name[ed + 1:]
    return name

def pack_result(human_detection, result, img_h, img_w):
    """Short summary.

    Args:
        human_detection (np.ndarray): Human detection result.
        result (type): The predicted label of each human proposal.
        img_h (int): The image height.
        img_w (int): The image width.

    Returns:
        tuple: Tuple of human proposal, label name and label score.
    """
    human_detection[:, 0::2] /= img_w
    human_detection[:, 1::2] /= img_h
    results = []
    if result is None:
        return None
    for prop, res in zip(human_detection, result):
        res.sort(key=lambda x: -x[1])
        results.append(
            (prop.data.cpu().numpy(), [x[0] for x in res], [x[1]
                                                            for x in res]))
    return results


def expand_bbox(bbox, h, w, ratio=1.25):
    x1, y1, x2, y2 = bbox
    center_x = (x1 + x2) // 2
    center_y = (y1 + y2) // 2
    width = x2 - x1
    height = y2 - y1

    square_l = max(width, height)
    new_width = new_height = square_l * ratio

    new_x1 = max(0, int(center_x - new_width / 2))
    new_x2 = min(int(center_x + new_width / 2), w)
    new_y1 = max(0, int(center_y - new_height / 2))
    new_y2 = min(int(center_y + new_height / 2), h)
    return (new_x1, new_y1, new_x2, new_y2)


def cal_iou(box1, box2):
    xmin1, ymin1, xmax1, ymax1 = box1
    xmin2, ymin2, xmax2, ymax2 = box2

    s1 = (xmax1 - xmin1) * (ymax1 - ymin1)
    s2 = (xmax2 - xmin2) * (ymax2 - ymin2)

    xmin = max(xmin1, xmin2)
    ymin = max(ymin1, ymin2)
    xmax = min(xmax1, xmax2)
    ymax = min(ymax1, ymax2)

    w = max(0, xmax - xmin)
    h = max(0, ymax - ymin)
    intersect = w * h
    union = s1 + s2 - intersect
    iou = intersect / union

    return iou


# clip_pose_extraction
def skeleton_based_stdet(predict_stepsize, video,
                         # skeleton_config, skeleton_stdet_checkpoint, device, action_score_thr, label_map,
                         human_detections, pose_results, num_frame, clip_len, frame_interval, h, w):
    window_size = clip_len * frame_interval
    assert clip_len % 2 == 0, 'We would like to have an even clip_len'
    timestamps = np.arange(window_size // 2, num_frame + 1 - window_size // 2,
                           predict_stepsize)

    # skeleton_config = mmengine.Config.fromfile(skeleton_config)
    # num_class = max(label_map.keys()) + 1  # for AVA dataset (81)
    # skeleton_config.model.cls_head.num_classes = num_class
    # skeleton_stdet_model = init_recognizer(skeleton_config,
    #                                        skeleton_stdet_checkpoint,
    #                                        device)

    skeleton_predictions = []
    skeleton_datasets = []

    print('Building skeleton datasets from existing keypoint data for each clip')
    prog_bar = mmengine.ProgressBar(len(timestamps))
    for timestamp in timestamps:  # iterate each clip
        proposal = human_detections[timestamp - 1] # get bboxes for persons in timestamp (first frame of clip)
        if proposal.shape[0] == 0:  # no people detected
            skeleton_predictions.append(None)
            continue

        start_frame = timestamp - (clip_len // 2 - 1) * frame_interval
        frame_inds = start_frame + np.arange(0, window_size, frame_interval)
        frame_inds = list(frame_inds - 1)
        num_frame = len(frame_inds)  # 30

        pose_result = [pose_results[ind] for ind in frame_inds]  # grouping frames poses for each clip

        skeleton_prediction = []
        for i in range(proposal.shape[0]):  # num_person  # iterate each bbox in timestamp (first frame of clip)
            skeleton_prediction.append([])

            fake_anno = dict(
                frame_dir=osp.splitext(osp.basename(video))[0]+"_"+str(timestamp+(i+1)*0.001),
                label=-1,
                img_shape=(h, w),
                original_shape=(h, w),
                num_clips=1,
                total_frames=num_frame
            )
            num_person = 1

            num_keypoint = 17
            keypoint = np.zeros(
                (num_person, num_frame, num_keypoint, 2))  # M T V 2
            keypoint_score = np.zeros(
                (num_person, num_frame, num_keypoint))  # M T V

            # pose matching
            person_bbox = proposal[i][:4]  # get bbox for a person in timestamp (first frame of clip)
            area = expand_bbox(person_bbox, h, w)  # bbox expanded by 1.25 ratio with square shape

            for j, poses in enumerate(pose_result):  # num_frame  # iterate each frame of clip
                max_iou = float('-inf')
                index = -1
                if len(poses['keypoints']) == 0:
                    continue
                for k, bbox in enumerate(poses['bboxes']):  # iterate each bbox/pose in each frame
                    iou = cal_iou(bbox, area)  # compare each bbox in each frame with current area (calculate_intersect/union)
                    if max_iou < iou:
                        index = k  # pose from the biggest intersect/union (iou) will be considered
                        max_iou = iou
                keypoint[0, j] = poses['keypoints'][index]
                keypoint_score[0, j] = poses['keypoint_scores'][index]

            fake_anno['keypoint'] = keypoint
            fake_anno['keypoint_score'] = keypoint_score

            skeleton_datasets.append(fake_anno)
            # output = inference_recognizer(skeleton_stdet_model, fake_anno)
            # # for multi-label recognition
            # score = output.pred_score.tolist()
            # for k in range(len(score)):  # 81
            #     if k not in label_map:
            #         continue
            #     if score[k] > action_score_thr:
            #         skeleton_prediction[i].append((label_map[k], score[k]))
            skeleton_prediction[i].append(("annotate!", timestamp + (i+1)*0.001))

        skeleton_predictions.append(skeleton_prediction)
        prog_bar.update()

    return timestamps, skeleton_predictions, skeleton_datasets

In [5]:
def hex2color(h):
    """Convert the 6-digit hex string to tuple of 3 int value (RGB)"""
    return (int(h[:2], 16), int(h[2:4], 16), int(h[4:], 16))

PLATEBLUE = '03045e-023e8a-0077b6-0096c7-00b4d8-48cae4'
PLATEBLUE = PLATEBLUE.split('-')
PLATEBLUE = [hex2color(h) for h in PLATEBLUE]


def visualize(pose_config,
              frames,
              annotations,
              pose_data_samples,
              action_result,
              plate=PLATEBLUE,
              max_num=5):
    """Visualize frames with predicted annotations.

    Args:
        frames (list[np.ndarray]): Frames for visualization, note that
            len(frames) % len(annotations) should be 0.
        annotations (list[list[tuple]]): The predicted spatio-temporal
            detection results.
        pose_data_samples (list[list[PoseDataSample]): The pose results.
        action_result (str): The predicted action recognition results.
        pose_model (nn.Module): The constructed pose model.
        plate (str): The plate used for visualization. Default: PLATEBLUE.
        max_num (int): Max number of labels to visualize for a person box.
            Default: 5.

    Returns:
        list[np.ndarray]: Visualized frames.
    """

    assert max_num + 1 <= len(plate)
    frames_ = cp.deepcopy(frames)
    frames_ = [mmcv.imconvert(f, 'bgr', 'rgb') for f in frames_]
    nf, na = len(frames), len(annotations)
    assert nf % na == 0
    nfpa = len(frames) // len(annotations)
    anno = None
    h, w, _ = frames[0].shape
    scale_ratio = np.array([w, h, w, h])

    # add pose results
    if pose_data_samples:
        pose_config = mmengine.Config.fromfile(pose_config)
        visualizer = VISUALIZERS.build(pose_config.visualizer | {'line_width':5, 'bbox_color':(101,193,255), 'radius': 8})  # https://mmpose.readthedocs.io/en/latest/api.html#mmpose.visualization.PoseLocalVisualizer
        visualizer.set_dataset_meta(pose_data_samples[0].dataset_meta)
        for i, (d, f) in enumerate(zip(pose_data_samples, frames_)):
            visualizer.add_datasample(
                'result',
                f,
                data_sample=d,
                draw_gt=False,
                draw_heatmap=False,
                draw_bbox=True,
                draw_pred=True,
                show=False,
                wait_time=0,
                out_file=None,
                kpt_thr=0.3)
            frames_[i] = visualizer.get_image()

    for i in range(na):
        anno = annotations[i]
        if anno is None:
            continue
        for j in range(nfpa):
            ind = i * nfpa + j
            frame = frames_[ind]

            # add spatio-temporal action detection results
            for ann in anno:
                box = ann[0]
                label = ann[1]
                if not len(label):
                    continue
                score = ann[2]
                box = (box * scale_ratio).astype(np.int64)
                st, ed = tuple(box[:2]), tuple(box[2:])
                if not pose_data_samples:
                    cv2.rectangle(frame, st, ed, plate[0], 2)

                for k, lb in enumerate(label):
                    if k >= max_num:
                        break
                    text = abbrev(lb)
                    text = ': '.join([text, f'{score[k]:.3f}'])
                    location = (0 + st[0], 18 + k * 18 + st[1])
                    textsize = cv2.getTextSize(text, FONTFACE, FONTSCALE,
                                               THICKNESS)[0]
                    textwidth = textsize[0]
                    diag0 = (location[0] + textwidth, location[1] - 14)
                    diag1 = (location[0], location[1] + 2)
                    cv2.rectangle(frame, diag0, diag1, plate[k + 1], -1)
                    FONTCOLOR = (255, 0, 0)
                    cv2.putText(frame, text, location, FONTFACE, FONTSCALE,
                                FONTCOLOR, THICKNESS, LINETYPE)

    return frames_

In [6]:
# video_folder_path = osp.abspath("/media/ciis/680d5156-2214-4b41-baf0-6bb993bff707/ciis-compnew/Documents/ActionTracking/video/")
extracted_folder_path = r"C:\Users\blute\Videos\skripsi\video\25_5_v1\sampleFrame_4_out_16fps"
video_folder_path = "../assets/video/25_5_v1"

excluded_file = ['lab_1','lab_2']
final_files = sorted([
        f for f in os.listdir(extracted_folder_path) 
        if os.path.splitext(f)[0] not in excluded_file
    ])


#if only 1 video
final_files = ['lab_7']

print(str(final_files))
for file in final_files:
    print(f"{extracted_folder_path}/{file}") 


['lab_7']
C:\Users\blute\Videos\skripsi\video\25_5_v1\sampleFrame_4_out_16fps/lab_7


In [7]:
# human detection config
det_config = "../mmaction2/demo/demo_configs/faster-rcnn_r50_fpn_2x_coco_infer.py"
det_checkpoint = 'http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth'
det_score_thr = 0.9
#det_cat_id = 0

# pose estimation config
pose_config = '../mmaction2/demo/demo_configs/td-hm_hrnet-w32_8xb64-210e_coco-256x192_infer.py'
pose_checkpoint = 'https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth'

# use skeleton-based method
use_skeleton_stdet = True
use_skeleton_recog = True

# skeleton-based spatio-temporal action classification config
label_map_stdet = "../mmaction2/tools/data/ciis/ciis_label_map.txt"

predict_stepsize = 2  # must even int, give out a spatio-temporal detection prediction per n frames
output_stepsize = 1  # show one frame per n frames in the demo, we should have: predict_stepsize % output_stepsize == 0, speedUp/slowDown video output
output_fps = 6  # the fps of demo video output, will speedUp/slowDown video output, must equal to (video_input_fps/output_stepsize) to get normal speed

device = 'cuda'

In [ ]:
# FILE processing template

num_ver = 3

path_extractPose = f'../dataset/2025/2025_{num_ver}/extracted_pose'
path_annData = f'../dataset/2025/2025_{num_ver}/annotated_data'
Path(path_extractPose).mkdir(parents=True, exist_ok=True)
Path(path_annData).mkdir(parents=True, exist_ok=True)

for folder in final_files:
    vidio_path = f'/{folder}'
    target_dir = str(extracted_folder_path) + str(vidio_path)
    video = str(video_folder_path) + str(vidio_path) + '.mp4'


    ann_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.csv'
    pkl_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.pkl'
    out_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '_gt.mp4'

    print(target_dir)
    print(video)
    print(pkl_filename)
    print(f'{extracted_folder_path}{vidio_path}')
    print(f'{extracted_folder_path}/STEP_{predict_stepsize}{vidio_path}.pkl')

C:\Users\blute\Videos\skripsi\video\25_5_v1\sampleFrame_4_out_16fps/lab_7
../assets/video/25_5_v1/lab_7.mp4
../dataset/2025/2025_3/extracted_pose/lab_7.pkl
C:\Users\blute\Videos\skripsi\video\25_5_v1\sampleFrame_4_out_16fps/lab_7
C:\Users\blute\Videos\skripsi\video\25_5_v1\sampleFrame_4_out_16fps/STEP_2/lab_7.pkl


# Chronically step 1

### Frame Extraction

In [ ]:
import os.path as osp
import os
from pathlib import Path
from mmaction.utils import frame_extract
import gc
import torch

extracted_folder_path = r"C:\Users\blute\Videos\skripsi\video\25_5_V2"
# video_folder_path = "../assets/video/25_5_v1"

excluded_file = ['h23_1']  # add if any
final_files = sorted([
    f for f in os.listdir(extracted_folder_path)
    if f.endswith('.mp4') and os.path.splitext(f)[0] not in excluded_file
])

print(str(final_files))

['h23_2.mp4', 'h23_3.mp4', 'h23_4.mp4', 'h23_5.mp4', 'h23_6.mp4', 'h23_7.mp4', 'sejajar_1.mp4', 'sejajar_2.mp4']


In [ ]:
for folder in final_files:
    folder_name = os.path.splitext(folder)[0]
    vidio_path = f'/{folder_name}'
    target_dir = str(extracted_folder_path)
    video = str(extracted_folder_path) + str(vidio_path) + '.mp4'
    # stdetPosePKL = f'{extracted_folder_path}/STEP_{predict_stepsize}{vidio_path}.pkl'
    # num_ver = 3

    # path_extractFrame = f'../dataset/2025/2025_{num_ver}/extracted_pose'
    # path_annData = f'../dataset/2025/2025_{num_ver}/annotated_data'
    Path(target_dir).mkdir(parents=True, exist_ok=True)
    # Path(path_annData).mkdir(parents=True, exist_ok=True)

    frame_paths, original_frames = frame_extract(
        video_path=video, 
        # 720,
        out_dir=target_dir)
    # num_frame = len(frame_paths)
    # h, w, _ = original_frames[0].shape

    # Clear cache and free up RAM after processing each video
    try:
        del frame_paths
        del original_frames
    except NameError:
        print("Some variables were not defined, skipping deletion.")

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

### Separate files in extracted frame folder for max n files 
*n depends on your ram size
> this process were made because dataset creation used so much ram for longer frame len

In [ ]:
import os
import shutil
import math

def split_files_into_folders(source_folder_path, max_files_per_folder=4500):
    """
    Memisahkan file dalam folder sumber ke beberapa subfolder secara sekuensial
    berdasarkan nama file, dengan jumlah file maksimal tertentu.

    Args:
        source_folder_path (str): Path ke folder sumber.
        max_files_per_folder (int): Jumlah maksimal file per subfolder.
    """
    try:
        if not os.path.isdir(source_folder_path):
            print(f"Error: Folder sumber '{source_folder_path}' tidak ditemukan.")
            return

        all_items = os.listdir(source_folder_path)
        files_to_move = [
            f for f in all_items if os.path.isfile(os.path.join(source_folder_path, f))
        ]

        if not files_to_move:
            print(f"Tidak ada file yang ditemukan di '{source_folder_path}'.")
            return

        # --- TAMBAHAN: Mengurutkan daftar file secara alphabetical ---
        files_to_move.sort()
        print("Daftar file telah diurutkan berdasarkan nama.")
        # -----------------------------------------------------------

        num_files = len(files_to_move)
        print(f"Total file yang ditemukan: {num_files}")

        num_subfolders = math.ceil(num_files / max_files_per_folder)
        print(f"Akan dibuat {num_subfolders} subfolder.")

        base_folder_name = os.path.basename(os.path.normpath(source_folder_path))
        parent_folder = os.path.dirname(source_folder_path) # Dapatkan folder induk

        for i in range(num_subfolders):
            subfolder_name = f"{base_folder_name}_{i + 1}"
            # Pastikan subfolder dibuat di lokasi yang sama dengan folder sumber
            subfolder_path = os.path.join(parent_folder, subfolder_name) 

            os.makedirs(subfolder_path, exist_ok=True)
            print(f"Subfolder '{subfolder_path}' telah dibuat/ditemukan.")

            start_index = i * max_files_per_folder
            end_index = start_index + max_files_per_folder

            files_for_current_subfolder = files_to_move[start_index:end_index]

            for file_name in files_for_current_subfolder:
                source_file_path = os.path.join(source_folder_path, file_name)
                destination_file_path = os.path.join(subfolder_path, file_name)

                try:
                    shutil.move(source_file_path, destination_file_path)
                except Exception as e:
                    print(f"  Error saat memindahkan '{file_name}': {e}")
            
            print(f"Berhasil memindahkan {len(files_for_current_subfolder)} file ke '{subfolder_name}'.")

        print("\nProses pemisahan file selesai.")

    except Exception as e:
        print(f"Terjadi kesalahan: {e}")


#========Process===========
n =4500
folder_path_input = input("Masukkan path ke folder yang ingin dipisah filenya: ")
split_files_into_folders(folder_path_input, n)

Daftar file telah diurutkan berdasarkan nama.
Total file yang ditemukan: 5325
Akan dibuat 2 subfolder.
Subfolder 'C:\Users\blute\Videos\skripsi\video\25_5_V2\h23_7_1' telah dibuat/ditemukan.
Berhasil memindahkan 4500 file ke 'h23_7_1'.
Subfolder 'C:\Users\blute\Videos\skripsi\video\25_5_V2\h23_7_2' telah dibuat/ditemukan.
Berhasil memindahkan 825 file ke 'h23_7_2'.

Proses pemisahan file selesai.


### Make annotation files
> *manually change the dataset version `num_ver`

In [ ]:
num_ver = 4
path_extractPose = f'../dataset/2025/2025_{num_ver}/extracted_pose'
path_annData = f'../dataset/2025/2025_{num_ver}/annotated_data'
Path(path_extractPose).mkdir(parents=True, exist_ok=True)
Path(path_annData).mkdir(parents=True, exist_ok=True)

for folder in final_files:
    print(f"\n--- Memulai proses untuk folder: {folder} ---") # Menambah indikator awal loop
    vidio_path = f'/{folder}'
    target_dir = str(extracted_folder_path) + str(vidio_path)
    video = str(video_folder_path) + str(vidio_path) + '.mp4'

    ann_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.csv'
    pkl_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.pkl'
    out_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '_gt.mp4'

    if not osp.exists(target_dir):
        print(f"Error: Direktori tidak ada: {target_dir}")
        continue # Lanjut ke folder berikutnya jika direktori tidak ada
    else:
        search_pattern = osp.join(target_dir, 'img_*.jpg')
        frame_paths = glob.glob(search_pattern)
        frame_paths.sort()

        if frame_paths:
            print(f"Ditemukan {len(frame_paths)} frame di '{target_dir}'.")
        else:
            print(f"Tidak ada frame 'img_*.jpg' ditemukan di '{target_dir}'.")
            continue # Lanjut ke folder berikutnya jika tidak ada frame

    num_frame = len(frame_paths)
    try:
        original_frames = cv2.imread(frame_paths[0])
        h, w, _ = original_frames.shape
    except Exception as e:
        print(f"Error membaca frame pertama: {e}")
        continue # Lanjut ke folder berikutnya jika error

    print("Memulai deteksi manusia...")
    human_detections, _ = detection_inference(
        det_config,
        det_checkpoint,
        frame_paths,
        det_score_thr,
        device=device)
    torch.cuda.empty_cache() # Membersihkan cache setelah deteksi
    print("Deteksi manusia selesai.")

    print("Memulai estimasi pose...")
    pose_datasample = None
    pose_results, pose_datasample = pose_inference(
        pose_config,
        pose_checkpoint,
        frame_paths,
        human_detections,
        device=device)
    torch.cuda.empty_cache() # Membersihkan cache setelah pose
    print("Estimasi pose selesai.")

    stdet_preds = None

    print('Memulai deteksi aksi SpatioTemporal berbasis skeleton...')
    clip_len, frame_interval = predict_stepsize, 1

    timestamps, stdet_preds, skeleton_datasets = skeleton_based_stdet(
        predict_stepsize, video,
        human_detections,
        pose_results, num_frame,
        clip_len,
        frame_interval, h, w)
    
    # Normalisasi (jika diperlukan, sesuaikan dengan kode asli Anda)
    for i in range(len(human_detections)):
        det = human_detections[i]
        det[:, 0:4:2] *= 1
        det[:, 1:4:2] *= 1
        human_detections[i] = torch.from_numpy(det[:, :4]).to(device)

    print("Menyimpan anotasi dan data skeleton...")
    anno = ""
    if stdet_preds:
        for clip in stdet_preds:
            if clip is None:
                continue
            for person_attr in clip:
                anno += str(person_attr[0][0]) + "," + str(person_attr[0][1]) + "\n"

    with open(ann_filename, 'w') as data:
        data.write(anno)

    mmengine.dump(skeleton_datasets, pkl_filename)
    print("Penyimpanan selesai.")

    print("Membuat video hasil...")
    stdet_results = []
    if stdet_preds:
        for timestamp, prediction in zip(timestamps, stdet_preds):
            if timestamp - 1 < len(human_detections):
                human_detection = human_detections[timestamp - 1]
                stdet_results.append(
                    pack_result(human_detection, prediction, h, w))

    def dense_timestamps(timestamps, n):
        old_frame_interval = (timestamps[1] - timestamps[0]) if len(timestamps) > 1 else 0
        start = timestamps[0] - old_frame_interval / n * (n - 1) / 2
        new_frame_inds = np.arange(
            len(timestamps) * n) * old_frame_interval / n + start
        return new_frame_inds.astype(np.int64)

    dense_n = int(predict_stepsize / output_stepsize)
    # if timestamps: # Pastikan timestamps tidak kosong
    output_timestamps = dense_timestamps(timestamps, dense_n) + 1
    frames = [
        cv2.imread(frame_paths[timestamp - 1])
        for timestamp in output_timestamps if timestamp - 1 < len(frame_paths)
    ]
    
    pose_datasample_out = [
        pose_datasample[timestamp - 1] for timestamp in output_timestamps if timestamp - 1 < len(pose_datasample)
    ]

    vis_frames = visualize(pose_config, frames, stdet_results, pose_datasample_out, None)
    vid = mpy.ImageSequenceClip(vis_frames, fps=output_fps)
    vid.write_videofile(out_filename) # Tambah logger=None untuk output lebih bersih
    print(f"Video hasil disimpan di: {out_filename}")


    # --- PENAMBAHAN: Membersihkan Memori RAM dan GPU ---
    print(f"--- Membersihkan memori setelah folder: {folder} ---")
    
    # Hapus variabel yang tidak diperlukan lagi secara eksplisit (opsional tapi bisa membantu)
    del human_detections, pose_results, pose_datasample, stdet_preds, skeleton_datasets
    del stdet_results, frames, vis_frames, vid, original_frames, frame_paths
    
    # Membersihkan RAM (Garbage Collection)
    gc.collect() 
    print("Garbage collection dijalankan.")
    
    # Membersihkan Cache GPU (jika menggunakan PyTorch dan CUDA)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("Cache CUDA dibersihkan.")
    
    print(f"--- Selesai iterasi untuk folder: {folder} ---\n")

#### unused

In [ ]:
for folder in final_files:
    vidio_path = f'/{folder}'
    target_dir = str(extracted_folder_path) + str(vidio_path)
    video = str(video_folder_path) + str(vidio_path) + '.mp4'
    num_ver = 3

    path_extractPose = f'../dataset/2025/2025_{num_ver}/extracted_pose'
    path_annData = f'../dataset/2025/2025_{num_ver}/annotated_data'
    Path(path_extractPose).mkdir(parents=True, exist_ok=True)
    Path(path_annData).mkdir(parents=True, exist_ok=True)
        

    ann_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.csv'
    pkl_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '.pkl'
    out_filename = f'../dataset/2025/2025_{num_ver}/extracted_pose' + str(vidio_path) + '_gt.mp4'

    if not osp.exists(target_dir):
        print(f"Error: The specified directory does not exist: {target_dir}")
    else:
        # Create the search pattern to find all 'img_*.jpg' files
        # directly within the target_dir.
        search_pattern = osp.join(target_dir, 'img_*.jpg')

        # Use glob.glob to find all matching file paths
        frame_paths = glob.glob(search_pattern)

        # Sort the paths to ensure they are in frame order
        frame_paths.sort()

        # Print the results
        if frame_paths:
            print(f"Found {len(frame_paths)} frames in '{target_dir}':")
            # Print the first few and the last one as an example
            for path in frame_paths[:5]: # Print first 5
                print(path)
            if len(frame_paths) > 5:
                print("...")
                print(frame_paths[-1]) # Print the last one
        else:
            print(f"No frames matching 'img_*.jpg' were found in '{target_dir}'.")

    num_frame=len(frame_paths)
    original_frames = cv2.imread(frame_paths[0])
    #print(original_frames.shape)
    h, w, _ = original_frames.shape


    # get Human detection results
    human_detections, _ = detection_inference(
        det_config,
        det_checkpoint,
        frame_paths,
        det_score_thr,
        device=device)
    torch.cuda.empty_cache()

    # get Pose estimation results
    pose_datasample = None
    pose_results, pose_datasample = pose_inference(
        pose_config,
        pose_checkpoint,
        frame_paths,
        human_detections,
        device=device)
    torch.cuda.empty_cache()

    stdet_preds = None

    print('Use skeleton-based SpatioTemporal Action Detection')
    # clip_len, frame_interval = 30, 1
    clip_len, frame_interval = predict_stepsize, 1

    # clip_pose_extraction
    timestamps, stdet_preds, skeleton_datasets = skeleton_based_stdet(predict_stepsize, video,
                                                                    # skeleton_config,
                                                                    # skeleton_stdet_checkpoint,
                                                                    # device,
                                                                    # action_score_thr,
                                                                    # stdet_label_map,
                                                                    human_detections,
                                                                    pose_results, num_frame,
                                                                    clip_len,
                                                                    frame_interval, h, w)
    for i in range(len(human_detections)):
        det = human_detections[i]
        # det[:, 0:4:2] *= w_ratio
        # det[:, 1:4:2] *= h_ratio
        det[:, 0:4:2] *= 1
        det[:, 1:4:2] *= 1
        human_detections[i] = torch.from_numpy(det[:, :4]).to(device)

    anno = ""
    for clip in stdet_preds:
        if clip == None:
            continue
        for person_attr in clip:
            anno += str(person_attr[0][0]) + "," + str(person_attr[0][1]) + "\n"

    with open(ann_filename,'w') as data:
        data.write(anno)

    mmengine.dump(skeleton_datasets, pkl_filename)

    stdet_results = []
    for timestamp, prediction in zip(timestamps, stdet_preds):
        human_detection = human_detections[timestamp - 1]
        stdet_results.append(
            pack_result(human_detection, prediction, h, w))

    def dense_timestamps(timestamps, n):
        """Make it nx frames."""
        old_frame_interval = (timestamps[1] - timestamps[0])
        start = timestamps[0] - old_frame_interval / n * (n - 1) / 2
        new_frame_inds = np.arange(
            len(timestamps) * n) * old_frame_interval / n + start
        return new_frame_inds.astype(np.int64)

    dense_n = int(predict_stepsize / output_stepsize)
    output_timestamps = dense_timestamps(timestamps, dense_n) + 1
    # frames = [
    #     cv2.imread(frame_paths[timestamp - 1])
    #     for timestamp in output_timestamps
    # ]

    pose_datasample = [
        pose_datasample[timestamp - 1] for timestamp in output_timestamps
    ]
    mmengine.dump(
    {
        'stdet_results': stdet_results,
        'pose_datasample': pose_datasample,
        'output_timestamps': output_timestamps
    },
    f'{extracted_folder_path}{vidio_path}.pkl'
)

    print(f"Finished processing {folder}.")

    # --------------------------------------------------
    # --- 🧹 START: Memory Clearing Section ---
    # --------------------------------------------------
    print("Clearing memory for next iteration...")

    # 1. Explicitly delete large variables from this iteration
    try:
        del frame_paths
        del original_frames
        del human_detections
        del pose_results
        del pose_datasample
        del stdet_preds
        del skeleton_datasets
        # del stdet_results # Uncomment if you define this
        # del anno # Uncomment if you define this
    except NameError:
        print("Some variables were not defined, skipping deletion.")

    # 2. Suggest Python's garbage collector to run (clears RAM)
    gc.collect()

    # 3. Clear PyTorch's unused cached GPU memory (clears GPU RAM)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Memory cleared.")
    # --- 🧹 END: Memory Clearing Section ---
    # --------------------------------------------------


Found 10239 frames in '../dataset/2025/sampleFrame_4_out_16fps/lab_3':
../dataset/2025/sampleFrame_4_out_16fps/lab_3/img_000001.jpg
../dataset/2025/sampleFrame_4_out_16fps/lab_3/img_000002.jpg
../dataset/2025/sampleFrame_4_out_16fps/lab_3/img_000003.jpg
../dataset/2025/sampleFrame_4_out_16fps/lab_3/img_000004.jpg
../dataset/2025/sampleFrame_4_out_16fps/lab_3/img_000005.jpg
...
../dataset/2025/sampleFrame_4_out_16fps/lab_3/img_010239.jpg
Loads checkpoint by http backend from path: http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_bbox_mAP-0.384_20200504_210434-a5d8aa15.pth
Performing Human Detection for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>] 10239/10239, 18.6 task/s, elapsed: 551s, ETA:     0s
Loads checkpoint by http backend from path: https://download.openmmlab.com/mmpose/top_down/hrnet/hrnet_w32_coco_256x192-c78dce93_20200708.pth
Performing Human Pose Estimation for each frame
[>>>>>>>>>>>>>>>>>>>>>>>>] 10239/10239, 15.3

# Add Label to Dataset - Step 2

In [13]:
import csv

def load_label_map(file_path):
    """Load Label Map.

    Args:
        file_path (str): The file path of label map.

    Returns:
        dict: The label map (label name -> int).
    """
    lines = open(file_path).readlines()
    lines = [x.strip().split(': ') for x in lines]
    return {x[1]: int(x[0]) for x in lines}

stdet_label_map = load_label_map(label_map_stdet)

stdet_label_map

{'berdiri': 0,
 'berjalan': 1,
 'berjongkok': 2,
 'merayap': 3,
 'melempar': 4,
 'membidik senapan': 5,
 'membidik pistol': 6,
 'memukul': 7,
 'menendang': 8,
 'menusuk': 9}

In [ ]:
dataset_dir_path = r"C:\Users\blute\Videos\skripsi\video\annotated"
final_files = sorted([
    os.path.splitext(f)[0]
    for f in os.listdir(dataset_dir_path)
    if f.endswith('.pkl') and os.path.splitext(f)[0] not in excluded_file
])

print(str(final_files))
for file in final_files:
    print(f"{dataset_dir_path}/{file}") 

['h23_1', 'h23_2', 'h23_3', 'h23_4_1', 'h23_4_2', 'h23_5_1', 'h23_5_2', 'h23_6', 'h23_7_1', 'h23_7_2', 'lab_4_1', 'lab_4_2', 'lab_4_3', 'lab_5_1', 'lab_5_2', 'sejajar_1', 'sejajar_2']
C:\Users\blute\Videos\skripsi\video\annotated/h23_1
C:\Users\blute\Videos\skripsi\video\annotated/h23_2
C:\Users\blute\Videos\skripsi\video\annotated/h23_3
C:\Users\blute\Videos\skripsi\video\annotated/h23_4_1
C:\Users\blute\Videos\skripsi\video\annotated/h23_4_2
C:\Users\blute\Videos\skripsi\video\annotated/h23_5_1
C:\Users\blute\Videos\skripsi\video\annotated/h23_5_2
C:\Users\blute\Videos\skripsi\video\annotated/h23_6
C:\Users\blute\Videos\skripsi\video\annotated/h23_7_1
C:\Users\blute\Videos\skripsi\video\annotated/h23_7_2
C:\Users\blute\Videos\skripsi\video\annotated/lab_4_1
C:\Users\blute\Videos\skripsi\video\annotated/lab_4_2
C:\Users\blute\Videos\skripsi\video\annotated/lab_4_3
C:\Users\blute\Videos\skripsi\video\annotated/lab_5_1
C:\Users\blute\Videos\skripsi\video\annotated/lab_5_2
C:\Users\blute

### Preprocess Data
>Edit 'annotate!' to 'none'


In [21]:
for file in final_files:

    anned_filename = str(Path(dataset_dir_path) / f"{file}_edited.csv")


    with open(anned_filename, newline='') as csvfile:
        spamreader = csv.reader(csvfile, delimiter=',')
        updated_rows = []
        for row in spamreader:
            if row[0] == 'annotate!':
                row[0] = 'none'
            updated_rows.append(row)

    with open(anned_filename, 'w', newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(updated_rows)

## Processing Dataset

In [22]:
for file in final_files:
    num_ver = 5
    path_extractPose = f'../dataset/2025/2025_{num_ver}/extracted_pose'
    path_annData = f'../dataset/2025/2025_{num_ver}/annotated_data'
    Path(path_extractPose).mkdir(parents=True, exist_ok=True)
    Path(path_annData).mkdir(parents=True, exist_ok=True)

    anned_filename = str(Path(dataset_dir_path) / f"{file}_edited.csv")
    pkl_filename = str(Path(dataset_dir_path) / f"{file}.pkl")

    pkl_final = str(Path(path_annData) / f"{file}.pkl")

    custom_annos = []

    with open(anned_filename, newline='') as csvfile:
        spamreader = csv.reader(csvfile, delimiter=',')
        for row in spamreader:
            if row[0] == 'none':
                continue
            label = stdet_label_map[row[0]]
            id = float(row[1])
            custom_annos.append([id, label])

    skeleton_datasets = mmengine.load(pkl_filename)

    custom_dataset = []
    
    for index, ann in enumerate(custom_annos):

        fake_anno = dict(
            frame_dir=osp.splitext(osp.basename(pkl_filename))[0]+"_"+str(ann[0]))

        for j, data in enumerate(skeleton_datasets):
            if fake_anno['frame_dir'] == data['frame_dir']:
                fake_anno['frame_dir'] += '_' + str(index)
                fake_anno['label'] = ann[1]
                fake_anno['img_shape'] = data['img_shape']
                fake_anno['original_shape'] = data['original_shape']
                fake_anno['num_clips'] = 1
                fake_anno['total_frames'] = data['total_frames']
                fake_anno['clip_len'] = data['total_frames']
                fake_anno['keypoint'] = data['keypoint']
                fake_anno['keypoint_score'] = data['keypoint_score']

        custom_dataset.append(fake_anno)

    mmengine.dump(custom_dataset, pkl_final)
    print(f'{file} has been processsed and saved to {pkl_final}')

h23_1 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\h23_1.pkl
h23_2 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\h23_2.pkl
h23_3 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\h23_3.pkl
h23_4_1 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\h23_4_1.pkl
h23_4_2 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\h23_4_2.pkl
h23_5_1 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\h23_5_1.pkl
h23_5_2 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\h23_5_2.pkl
h23_6 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\h23_6.pkl
h23_7_1 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\h23_7_1.pkl
h23_7_2 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\h23_7_2.pkl
lab_4_1 has been processsed and saved to ..\dataset\2025\2025_5\annotated_data\lab_4_1.pkl
lab_4_2 has bee

# Visualize - Optional Step
> do it in clip_pose_extraction.ipynb

# Combine PKL - Final Step

In [ ]:
import os
import re

In [ ]:
num_ver = 5
path_annData = f'../dataset/2025/2025_{num_ver}/annotated_data'


split_ratio = 0.8
split_str = str(split_ratio).replace(".", "s")
base_dir = '../dataset/2025/train_dataset/'
Path(base_dir).mkdir(parents=True, exist_ok=True)

pattern = f'ciis21_{split_str}_v'
ext = '.pkl'

# Auto Versioning
versions = [
    int(m.group(1)) for f in os.listdir(base_dir)
    if (m := re.match(fr'{pattern}(\d+){re.escape(ext)}', f))
]
next_version = max(versions, default=0) + 1

# Final file path
combined_pkl = os.path.join(base_dir, f'{pattern}{next_version}{ext}')
print(combined_pkl)

../dataset/2025/train_dataset/ciis_0s8_v1.pkl


In [37]:
custom_datasets = dict(split=dict(xsub_train=[],
                                  xsub_val=[],
                                  xview_train=[],
                                  xview_val=[]),
                       annotations=[])

In [ ]:
for file in os.listdir(path_annData):
    print(file)
    if not file.endswith('.pkl'):
        continue
    custom_dataset = mmengine.load(os.path.join(path_annData, file))
    for i, data in enumerate(custom_dataset):
        custom_datasets['annotations'].append(data)
        if (i % 10) < (split_ratio * 10):
            custom_datasets['split']['xsub_train'].append(data['frame_dir'])
            custom_datasets['split']['xview_train'].append(data['frame_dir'])
        else:
            custom_datasets['split']['xsub_val'].append(data['frame_dir'])
            custom_datasets['split']['xview_val'].append(data['frame_dir'])

mmengine.dump(custom_datasets, combined_pkl)
print(f"new dataset save to {combined_pkl}")

h23_1.pkl
h23_2.pkl
h23_3.pkl
h23_4_1.pkl
h23_4_2.pkl
h23_5_1.pkl
h23_5_2.pkl
h23_6.pkl
h23_7_1.pkl
h23_7_2.pkl
lab_4_1.pkl
lab_4_2.pkl
lab_4_3.pkl
lab_5_1.pkl
lab_5_2.pkl
sejajar_1.pkl
sejajar_2.pkl
